# Predictive Road Safety on Colab

Thin runner (design doc §30): all logic lives in the repo; this notebook only clones it, installs
it and calls the scripts.

- Raw shards and the processed cache live on this machine's local disk, never on Drive.
- Runs (checkpoints, logs, metrics) go to Drive, so a disconnected session can resume.
- Use a GPU runtime (Runtime → Change runtime type).

**Smoke test (end of Stage 6):** run every cell once on the `dev` tier. Then disconnect and delete
the runtime, reconnect, rerun the setup cells, and run only the resume cell. The run should
continue from its last epoch.

In [ ]:
# Settings
REPO = "github.com/<your-user>/road-predict"  # private repo
BRANCH = "main"
CONFIG = "configs/lstm_map.yaml"
DATA_CONFIG = "configs/data_dev.yaml"
RUN_NAME = "colab_smoke"
RUNS_DIR = "/content/drive/MyDrive/road-predict/runs"
WORKDIR = "/content/road-predict"

## Setup (rerun after every reconnect)

In [ ]:
from google.colab import auth, drive, userdata

drive.mount("/content/drive")
auth.authenticate_user()  # the Google account registered for WOMD, for the GCS bucket

In [ ]:
import os
import subprocess

# A GitHub token with read access to the private repo, stored as the Colab secret GITHUB_TOKEN.
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists(WORKDIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, f"https://{token}@{REPO}.git", WORKDIR], check=True
    )
subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)
%cd {WORKDIR}
!git log --oneline -1

In [ ]:
# Keep Colab's preinstalled CUDA build of torch: its version satisfies torch>=2.5, so pip leaves it.
!pip install -q -e ".[ingest,dev]"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.__version__, torch.cuda.is_available())

## Data: download to local disk, build the cache

In [ ]:
!python scripts/download_shards.py --config {DATA_CONFIG}
!python scripts/preprocess_waymo.py --config {DATA_CONFIG}

In [ ]:
# Fast checks on this machine (the MPS tests skip themselves).
!python -m pytest -m "not slow" -q

## Train (checkpoints to Drive every epoch)

In [ ]:
!python scripts/train.py --config {CONFIG} --data-config {DATA_CONFIG} \
    --name {RUN_NAME} --out-dir {RUNS_DIR} --epochs 2

## Resume (after a disconnect: rerun Setup and Data, then this cell)

In [ ]:
!python scripts/train.py --resume {RUNS_DIR}/{RUN_NAME} --epochs 3

In [ ]:
import json

metrics = json.load(open(f"{RUNS_DIR}/{RUN_NAME}/metrics.json"))
print("device:", metrics["device"], metrics.get("gpu_name"))
print("epochs done:", len(metrics["history"]), "| resumed:", metrics["run"].get("resumed"))